In [1]:
# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# operation fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap. Prediction and optimization run one horizon at a time, while
# each LightGBM model uses native threads, so memory remains bounded.
import os, sys
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *
install_memory_guard()


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 8.2G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


14298148864

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import psutil
from tqdm import tqdm

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.append(os.path.abspath("../2_Features_build")) ; import target_features

from pathlib import Path
import joblib
from sklearn.isotonic import IsotonicRegression


Inputs

In [3]:
SELECTED_FEATURES_DIR = variables.CWD / "4_Features_select" / "Selected_features"
FEATURE_OBJECTIVES = ["normal", "arcsinh", "spikes", "dips"]
features_optimal_amount_by_objective = {
    objective: pd.read_parquet(SELECTED_FEATURES_DIR / f"FEATURES_OPTIMAL_AMOUNT_{objective}.parquet")
    for objective in FEATURE_OBJECTIVES
}

# The full matrix is ~10 GB and won't fit in RAM; only the selected features are
# used at predict time, so read the union of the selected columns instead.
_horizons = [f"h{i}" for i in range(1, variables.HORIZON_COUNT + 1)]
needed_features = sorted({
    feat
    for optimal in features_optimal_amount_by_objective.values()
    for h in _horizons
    for feat in optimal.loc[optimal[h] == True, "feature"].tolist()
})

features = read_parquet_float32(variables.FEATURES_DATASET_PATH, columns=needed_features)
targets  = pd.read_parquet(variables.AGG_TARGET_DATASET_PATH)


Loading..: 100%|██████████| 133/133 [00:13<00:00,  9.54batch/s]


In [4]:
# Match notebook 2: do not fork model workers. Run one prediction task at a
# time and give the active LightGBM model most physical CPU cores.
THREADS_PER_MODEL = max(1, psutil.cpu_count(logical=False) - 2)

trained_models = []
for path in sorted(Path(variables.TRAINED_MODELS_PATH).glob("*.joblib")):
    filename = path.stem
    parts = filename.split("_", 1)
    horizon = int(parts[0][1:])
    model_name = parts[1]
    model = joblib.load(path)
    model.set_params(n_jobs=THREADS_PER_MODEL, num_threads=THREADS_PER_MODEL)
    trained_models.append({"horizon": horizon, "model_name": model_name, "model": model})

print(f"Loaded {len(trained_models)} models; {THREADS_PER_MODEL} threads per active model")
display(trained_models[:2])


Loaded 8 models; 10 threads per active model


[{'horizon': 1,
  'model_name': 'full_range_regressor_clipped_MAE_loss',
  'model': LGBMRegressor(bagging_fraction=0.85, bagging_freq=5, feature_fraction=0.8,
                force_col_wise=True, learning_rate=0.025, max_bin=127,
                metric='mae', min_child_samples=25, n_estimators=1400, n_jobs=10,
                num_leaves=95, num_threads=10, objective='regression_l1',
                path_smooth=0.1, random_state=42, reg_alpha=0.05, reg_lambda=0.2,
                verbose=-1)},
 {'horizon': 1,
  'model_name': 'full_range_regressor_unclipped_RMSE_loss',
  'model': LGBMRegressor(bagging_fraction=0.85, bagging_freq=5, feature_fraction=0.8,
                force_col_wise=True, learning_rate=0.025, max_bin=127,
                metric='rmse', min_child_samples=25, n_estimators=1200, n_jobs=10,
                num_leaves=95, num_threads=10, objective='regression',
                path_smooth=0.1, random_state=43, reg_alpha=0.05, reg_lambda=0.2,
                verbose=-1)}]

In [5]:
def build_model_dict(trained_models):
    horizons = sorted(set(m["horizon"] for m in trained_models))
    mm = {(m["horizon"], m["model_name"]): m["model"] for m in trained_models}
    return {
        "full_range_regressor_clipped_MAE_loss": [mm.get((h, "full_range_regressor_clipped_MAE_loss")) for h in horizons],
        "full_range_regressor_unclipped_RMSE_loss": [mm.get((h, "full_range_regressor_unclipped_RMSE_loss")) for h in horizons],
        "positive_spike_classifier_binary_loss": [mm.get((h, "positive_spike_classifier_binary_loss")) for h in horizons],
        "positive_spike_regressor_unclipped_mae_loss": [mm.get((h, "positive_spike_regressor_unclipped_mae_loss")) for h in horizons],
        "positive_spike_regressor_unclipped_quantile_loss": [mm.get((h, "positive_spike_regressor_unclipped_quantile_loss")) for h in horizons],
        "negative_spike_classifer_unclipped_binary_loss": [mm.get((h, "negative_spike_classifer_unclipped_binary_loss")) for h in horizons],
        "negative_spike_regressor_unclipped_mae_loss": [mm.get((h, "negative_spike_regressor_unclipped_mae_loss")) for h in horizons],
        "negative_spike_regressor_unclipped_quantile_loss": [mm.get((h, "negative_spike_regressor_unclipped_quantile_loss")) for h in horizons],
        "horizon_list":  horizons,
    }

model_dict = build_model_dict(trained_models)

pd.DataFrame(model_dict)[:3]

,full_range_regressor_clipped_MAE_loss,full_range_regressor_unclipped_RMSE_loss,positive_spike_classifier_binary_loss,positive_spike_regressor_unclipped_mae_loss,positive_spike_regressor_unclipped_quantile_loss,negative_spike_classifer_unclipped_binary_loss,negative_spike_regressor_unclipped_mae_loss,negative_spike_regressor_unclipped_quantile_loss,horizon_list
0,"LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMClassifier(bagging_fraction=0.85, bagging_...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(alpha=0.9, bagging_fraction=0.85...","LGBMClassifier(bagging_fraction=0.85, bagging_...","LGBMRegressor(bagging_fraction=0.85, bagging_f...","LGBMRegressor(alpha=0.1, bagging_fraction=0.85...",1


In [6]:
X_validate = features[(features.index > variables.VALID_START) & (features.index <= variables.TEST_START)]
y_validate = targets[(targets.index > variables.VALID_START) & (targets.index <= variables.TEST_START)]

horizon_list = model_dict["horizon_list"]
full_range_regressor_clipped_MAE_loss = model_dict["full_range_regressor_clipped_MAE_loss"]
full_range_regressor_unclipped_RMSE_loss = model_dict["full_range_regressor_unclipped_RMSE_loss"]
positive_spike_classifier_binary_loss = model_dict["positive_spike_classifier_binary_loss"]
positive_spike_regressor_unclipped_mae_loss = model_dict["positive_spike_regressor_unclipped_mae_loss"]
positive_spike_regressor_unclipped_quantile_loss = model_dict["positive_spike_regressor_unclipped_quantile_loss"]
negative_spike_classifer_unclipped_binary_loss = model_dict["negative_spike_classifer_unclipped_binary_loss"]
negative_spike_regressor_unclipped_mae_loss = model_dict["negative_spike_regressor_unclipped_mae_loss"]
negative_spike_regressor_unclipped_quantile_loss = model_dict["negative_spike_regressor_unclipped_quantile_loss"]

Helpers

In [7]:
def get_features_for_this_horizon(h, objectives=None):
    # One feature matrix per requested objective. Columns = that objective's selected
    # features + the horizon-h target-time block, matching the training layout
    # (target_features.py). Full-range models use normal/arcsinh, spike -> spikes, dip -> dips.
    objectives = objectives if objectives is not None else list(features_optimal_amount_by_objective)
    base = X_validate.reindex(y_validate.index)
    out = {}
    for objective in objectives:
        optimal = features_optimal_amount_by_objective[objective]
        feat_cols = optimal.loc[optimal[f"h{h}"] == True, "feature"].tolist()
        out[objective] = target_features.append_target_time_feats(base[feat_cols].astype(np.float32), h)
    return out


def convert_from_asinh(y):
    return np.sinh(y) * variables.PRICE_TRANSFORM_SCALE


def run_horizon_tasks(function, description):
    # Same memory-safe pattern as notebook 2: sequential tasks, native model threads.
    tasks = list(enumerate(horizon_list))
    return [function(task) for task in tqdm(tasks, desc=description, unit="h")]


Step 1) Calculate full range 'alpha' values

In [8]:
def _find_alpha_for_horizon(task):
    i, h = task
    X_validate_h = get_features_for_this_horizon(h, ["normal", "arcsinh"]) # Per-objective features
    y_validate_h = y_validate[f"target_h{h}"].values.astype(np.float32) # Get the targets for this horizon
    # Predictions
    pred_model_1 = convert_from_asinh(full_range_regressor_clipped_MAE_loss[i].predict(X_validate_h["normal"])).astype(np.float32)
    pred_model_2 = convert_from_asinh(full_range_regressor_unclipped_RMSE_loss[i].predict(X_validate_h["arcsinh"])).astype(np.float32)

    best_alpha = 0.0 ; best_mae = float("inf")

    for alpha in np.linspace(0.0, 10, 50, dtype=np.float32):
        prediction = ((1.0 - alpha) * pred_model_1 + alpha * pred_model_2)

        mae = float(np.mean(np.abs(y_validate_h - prediction)))

        if mae < best_mae:
            best_mae = mae
            best_alpha = float(alpha)

    return i, best_alpha

# results = run_horizon_tasks(_find_alpha_for_horizon, "Alpha per horizon")
# results = pd.DataFrame(results, columns=["Horizon", "Alpha"])
# results.to_csv(variables.FULL_RANGE_ALPHA_PATH, index=False)
# results[:100]


Step 2) Combine full range models based on alphas

In [9]:
def model_combiner_full_range_blend(task):
    i, h = task
    X_validate_h = get_features_for_this_horizon(h, ["normal", "arcsinh"]) # Per-objective features
    
    # Get the targets for this horizon
    # y_validate_h = y_validate[f"target_h{h}"].values.astype(np.float32)
    # Create X_validate_h_naive, which is just a single feature, which is just same price last week
    # X_validate_h_naive = X_validate_h.reindex(X_validate_h.index)[naive_baseline_predictor[variables.NAIVE_BASELINE_PREDICTOR_COLUMN]].values.astype(np.float32)
    
    y_validate_h_pred_model_1 = convert_from_asinh(full_range_regressor_clipped_MAE_loss[i].predict(X_validate_h["normal"])).astype(np.float32)  # Use MODEL 1, to Make predictions (now on the validation data) for this horizon 
    y_validate_h_pred_model_2 = convert_from_asinh(full_range_regressor_unclipped_RMSE_loss[i].predict(X_validate_h["arcsinh"]))  # Use MODEL 2, to Make predictions (now on the validation data) for this horizon

    alpha_small = (1.0 - horizon_full_range_blend_alphas[i]) ; alpha_large = horizon_full_range_blend_alphas[i]
    y_validate_h_pred_full_range_blend = (alpha_small * y_validate_h_pred_model_1 + alpha_large * y_validate_h_pred_model_2).astype(np.float32) # Actually apply the blend and merge the first two models predictions

    return pd.Series(y_validate_h_pred_full_range_blend, index = X_validate.index, name = f"h{h}")

# horizon_full_range_blend_alphas = pd.read_csv(variables.FULL_RANGE_ALPHA_PATH)["Alpha"]
# results = run_horizon_tasks(model_combiner_full_range_blend, "Full-range blend per horizon")
# results = pd.concat(results, axis=1)
# results.to_parquet(variables.FULL_RANGE_BLEND_PATH)
# results[:10]


Step 3) Define a function to combine all models

In [10]:
def all_model_combiner_per_h(h, spike_adjustment_mode, spike_source_mode, spike_probability_threshold, spike_probability_power, spike_max_uplift_weight, dip_adjustment_mode=0, dip_source_mode=0, dip_probability_threshold=0.0, dip_probability_power=0.0, dip_max_reduction_weight=0.0):
    """
    Combine all models: base → spike adjustment → dip adjustment
    """
    i = horizon_list.index(h)  # Convert horizon value to index
    X_validate_h = get_features_for_this_horizon(h, ["spikes", "dips"]) # Per-objective features
    y_pred = full_range_blend_by_horizon[f"h{h}"].to_numpy()
    
    # === SPIKE ADJUSTMENT ===
    y_pred_spike_probability = positive_spike_classifier_binary_loss[i].predict_proba(X_validate_h["spikes"])[:, 1].astype(np.float32)
    y_pred_spike_mae = convert_from_asinh(positive_spike_regressor_unclipped_mae_loss[i].predict(X_validate_h["spikes"])).astype(np.float32)
    y_pred_spike_quantile = convert_from_asinh(positive_spike_regressor_unclipped_quantile_loss[i].predict(X_validate_h["spikes"])).astype(np.float32)
    
    # Choose spike source
    if spike_source_mode == 0: 
        spike_source = y_pred_spike_quantile
    elif spike_source_mode == 1: 
        spike_source = np.maximum(y_pred_spike_mae, y_pred_spike_quantile).astype(np.float32)
    elif spike_source_mode == 2:
        spike_source = y_pred_spike_mae
    else:
        spike_source = y_pred_spike_mae
    
    # Apply spike adjustment
    if spike_adjustment_mode == 0: 
        probability_of_no_spike = (1.0 - y_pred_spike_probability)
        probability_weighted_spike_value = y_pred_spike_probability * spike_source
        y_pred = (probability_of_no_spike * y_pred + probability_weighted_spike_value).astype(np.float32)
    
    elif spike_adjustment_mode == 1: 
        spike_probability_above_threshold = y_pred_spike_probability - spike_probability_threshold
        max_spike_probability_above_threshold = 1.0 - spike_probability_threshold + 1e-6
        normalized_spike_probability_above_threshold = np.clip((spike_probability_above_threshold) / (max_spike_probability_above_threshold), 0.0, 1.0)
        shaped_spike_confidence = np.power(normalized_spike_probability_above_threshold, spike_probability_power)
        shaped_spike_confidence_weighted = (shaped_spike_confidence * spike_max_uplift_weight).astype(np.float32)
        positive_spike_gap = np.maximum(spike_source - y_pred, 0.0)
        weighted_spike_uplift = positive_spike_gap * shaped_spike_confidence_weighted
        y_pred = (y_pred + weighted_spike_uplift).astype(np.float32)
    
    elif spike_adjustment_mode == 2: 
        spike_probability_above_threshold_gate = y_pred_spike_probability >= spike_probability_threshold
        y_pred = y_pred.copy()
        base_predictions_at_gated_rows = y_pred[spike_probability_above_threshold_gate]
        shifted_base_at_gated_rows = base_predictions_at_gated_rows + float(spike_probability_power)
        spike_source_gap_at_gated_rows = spike_source[spike_probability_above_threshold_gate] - y_pred[spike_probability_above_threshold_gate]
        positive_spike_gap_at_gated_rows = np.maximum(spike_source_gap_at_gated_rows, 0.0)
        y_pred[spike_probability_above_threshold_gate] = shifted_base_at_gated_rows * positive_spike_gap_at_gated_rows
        y_pred = y_pred.astype(np.float32)
    
    # === DIP ADJUSTMENT ===
    y_pred_dip_probability = negative_spike_classifer_unclipped_binary_loss[i].predict_proba(X_validate_h["dips"])[:, 1].astype(np.float32)
    y_pred_dip_mae = convert_from_asinh(negative_spike_regressor_unclipped_mae_loss[i].predict(X_validate_h["dips"])).astype(np.float32)
    y_pred_dip_quantile = convert_from_asinh(negative_spike_regressor_unclipped_quantile_loss[i].predict(X_validate_h["dips"])).astype(np.float32)
    
    # Choose dip source
    if dip_source_mode == 0:
        dip_source = y_pred_dip_quantile
    elif dip_source_mode == 1:
        dip_source = np.minimum(y_pred_dip_mae, y_pred_dip_quantile).astype(np.float32)
    elif dip_source_mode == 2:
        dip_source = y_pred_dip_mae
    else:
        dip_source = y_pred_dip_mae
    
    # Apply dip adjustment
    if dip_adjustment_mode == 0:
        probability_of_no_dip = (1.0 - y_pred_dip_probability)
        probability_weighted_dip_value = y_pred_dip_probability * dip_source
        y_pred = (probability_of_no_dip * y_pred + probability_weighted_dip_value).astype(np.float32)
    
    elif dip_adjustment_mode == 1:
        dip_probability_above_threshold = y_pred_dip_probability - dip_probability_threshold
        max_dip_probability_above_threshold = 1.0 - dip_probability_threshold + 1e-6
        normalized_dip_probability_above_threshold = np.clip((dip_probability_above_threshold) / (max_dip_probability_above_threshold), 0.0, 1.0)
        shaped_dip_confidence = np.power(normalized_dip_probability_above_threshold, dip_probability_power)
        shaped_dip_confidence_weighted = (shaped_dip_confidence * dip_max_reduction_weight).astype(np.float32)
        negative_dip_gap = np.maximum(y_pred - dip_source, 0.0)
        weighted_dip_reduction = negative_dip_gap * shaped_dip_confidence_weighted
        y_pred = (y_pred - weighted_dip_reduction).astype(np.float32)
    
    return y_pred

full_range_blend_by_horizon = pd.read_parquet(variables.FULL_RANGE_BLEND_PATH)


Step 4a) Call step 3 over a grid search, over 5 variables , measure error, store best

In [11]:
def initial_benchmark_best_combiner_vars_per_h(task):
    i, h = task

    y_pred = all_model_combiner_per_h(h,0,0,0,0,0)
    y = y_validate[f"target_h{h}"].values.astype(np.float32)
    
    spike_mask = y > variables.SPIKE_THRESHOLD
    non_spike_mask = y < variables.SPIKE_THRESHOLD
    dip_mask = y < variables.DIP_THRESHOLD
    
    comp1_error = np.abs(np.mean(y[spike_mask] - y_pred[spike_mask]))
    comp2_error = np.abs(np.mean(y[non_spike_mask] - y_pred[non_spike_mask]))
    comp3_error = np.abs(np.mean(y[dip_mask] - y_pred[dip_mask]))
    comp4_error = np.abs(np.mean(y - y_pred))
    
    overall_error = 2.2 * comp1_error + 1.3 * comp3_error + 1.0 * comp2_error + 0.25 * comp4_error

    return [overall_error,0,0,0,0,0]


def grid_search_best_spike_combiner_vars_per_h(task):
    i, h = task

    grid_search_dict = {
        "spike_adjustment_modes" : np.array([0, 1, 2], dtype=np.float32),
        "spike_source_modes" : np.array([0, 1, 2], dtype=np.float32),
        "probability_thresholds" : np.array([0.005], dtype=np.float32), 
        "probability_powers" : np.array([18], dtype=np.float32),
        "max_uplift_weights" : np.array([5], dtype=np.float32)
    }

    y = y_validate[f"target_h{h}"].values.astype(np.float32)
    
    # Calculate baseline (no adjustment) prediction and error.
    y_pred_baseline = all_model_combiner_per_h(h, 0, 0, 0, 0, 0)
    non_spike_mask = y < variables.SPIKE_THRESHOLD
    baseline_non_spike_error = np.abs(np.mean(y[non_spike_mask] - y_pred_baseline[non_spike_mask]))

    best_params = baseline_results[i].copy()
    best_error = best_params[0]

    for a in grid_search_dict["spike_adjustment_modes"]:
        for b in grid_search_dict["spike_source_modes"]:
            for c in grid_search_dict["probability_thresholds"]:
                for d in grid_search_dict["probability_powers"]:
                    for e in grid_search_dict["max_uplift_weights"]:
                        
                        y_pred = all_model_combiner_per_h(h,a,b,c,d,e)

                        spike_mask = y > variables.SPIKE_THRESHOLD
                        non_spike_mask = y < variables.SPIKE_THRESHOLD
                        dip_mask = y < variables.DIP_THRESHOLD
                        
                        comp1_error = np.abs(np.mean(y[spike_mask] - y_pred[spike_mask]))
                        comp2_error = np.abs(np.mean(y[non_spike_mask] - y_pred[non_spike_mask]))
                        comp3_error = np.abs(np.mean(y[dip_mask] - y_pred[dip_mask]))
                        comp4_error = np.abs(np.mean(y - y_pred))
                    
                        if comp2_error > baseline_non_spike_error + 0.20:
                            continue
                        
                        overall_error = 2.2 * comp1_error + 1.3 * comp3_error + 1.0 * comp2_error + 0.25 * comp4_error
                           
                        if overall_error < best_error: 
                            best_error = overall_error
                            best_params = [overall_error, a, b, c, d, e]

    return best_params


# baseline_results = run_horizon_tasks(initial_benchmark_best_combiner_vars_per_h, "Baseline per horizon")
# spike_results = run_horizon_tasks(grid_search_best_spike_combiner_vars_per_h, "Spike grid per horizon")
# spike_results = pd.DataFrame(
#     spike_results,
#     columns=["overall_error", "spike_adjustment_mode", "spike_source_mode", "probability_threshold", "probability_power", "max_uplift_weight"],
#     index=horizon_list,
# )
# spike_results["baseline_error"] = [result[0] for result in baseline_results]
# spike_results.to_csv(variables.SPIKE_MODELS_BLEND_PARAMS_PATH)
# spike_results.head()

Step 4b)

In [12]:
def grid_search_best_dip_combiner_vars_per_h(task):
    i, h = task
    
    # Use the best spike parameters from previous step
    spike_params = spike_results.loc[h, ["spike_adjustment_mode", "spike_source_mode", "probability_threshold", "probability_power", "max_uplift_weight"]].values
    
    grid_search_dict = {
        "dip_adjustment_modes": np.array([0, 1], dtype=np.int8),
        "dip_source_modes": np.array([0, 1, 2], dtype=np.int8),
        "probability_thresholds": np.array([0.005], dtype=np.float32),
        "probability_powers": np.array([18], dtype=np.float32),
        "max_reduction_weights": np.array([5], dtype=np.float32)
    }
    
    y = y_validate[f"target_h{h}"].values.astype(np.float32)
    
    # Calculate initial error (spike adjustment only, no dip adjustment)
    y_pred_initial = all_model_combiner_per_h(h, *spike_params, 0, 0, 0.0, 0.0, 0.0)
    
    spike_flag = y > variables.SPIKE_THRESHOLD
    dip_flag = y < variables.DIP_THRESHOLD
    non_dip_flag = ~dip_flag
    
    initial_non_dip_error = np.abs(np.mean(y[non_dip_flag] - y_pred_initial[non_dip_flag]))
    
    # Initialize best parameters
    best_error = float("inf")
    best_params = [best_error, 0, 0, 0.0, 0.0, 0.0]
    
    # Grid search
    for a in grid_search_dict["dip_adjustment_modes"]:
        for b in grid_search_dict["dip_source_modes"]:
            for c in grid_search_dict["probability_thresholds"]:
                for d in grid_search_dict["probability_powers"]:
                    for e in grid_search_dict["max_reduction_weights"]:
                        
                        y_pred = all_model_combiner_per_h(h, *spike_params, a, b, c, d, e)
                        
                        dip_error = np.abs(np.mean(y[dip_flag] - y_pred[dip_flag])) if dip_flag.sum() > 0 else 0.0
                        spike_error = np.abs(np.mean(y[spike_flag] - y_pred[spike_flag])) if spike_flag.sum() > 0 else 0.0
                        non_dip_error = np.abs(np.mean(y[non_dip_flag] - y_pred[non_dip_flag])) if non_dip_flag.sum() > 0 else 0.0
                        all_error = np.abs(np.mean(y - y_pred))
                        
                        # Penalize if non-dip error increases by more than 20%
                        if non_dip_error > initial_non_dip_error + 0.20:
                            continue
                        
                        # Weighted overall error (emphasize dip and spike errors)
                        overall_error = 2.2 * dip_error + 1.3 * spike_error + 1.0 * non_dip_error + 0.25 * all_error
                        
                        if overall_error < best_error:
                            best_error = overall_error
                            best_params = [overall_error, int(a), int(b), float(c), float(d), float(e)]
    
    return best_params

# spike_results = pd.read_csv(variables.SPIKE_MODELS_BLEND_PARAMS_PATH, index_col=0)
# dip_results = run_horizon_tasks(grid_search_best_dip_combiner_vars_per_h, "Dip grid per horizon")
# dip_results = pd.DataFrame(
#     dip_results,
#     columns=["overall_error", "dip_adjustment_mode", "dip_source_mode", "probability_threshold", "probability_power", "max_reduction_weight"],
#     index=horizon_list,
# )
# dip_results.to_csv(variables.DIP_MODELS_BLEND_PARAMS_PATH)
# dip_results.head()

Step 5) Build isotonic regression calibrators using the optimal spike and dip parameters.

In practice: IsotonicRegression is a common final calibration step in ensemble pipelines to improve the accuracy of probabilistic predictions without changing the rank order of predictions. It's particularly useful when you have complex blending logic (like spike/dip adjustments) that might introduce calibration drift.



In [13]:
def build_calibrators(task):
    i, h = task

    # Get optimal parameters from grid search results
    spike_params = spike_results.loc[h, ["spike_adjustment_mode", "spike_source_mode", "probability_threshold", "probability_power", "max_uplift_weight"]].values
    dip_params = dip_results.loc[h, ["dip_adjustment_mode", "dip_source_mode", "probability_threshold", "probability_power", "max_reduction_weight"]].values
    
    # Create the final model prediction using the full pipeline
    y_pred = all_model_combiner_per_h(h, *spike_params, *dip_params)
    
    # Get the targets for this horizon
    y_actual = y_validate[f"target_h{h}"].values.astype(np.float32)
    
    # Fit isotonic calibrator
    calibrator = IsotonicRegression(out_of_bounds="clip")
    calibrator.fit(y_pred, y_actual)

    return calibrator

# spike_results = pd.read_csv(variables.SPIKE_MODELS_BLEND_PARAMS_PATH, index_col=0)
# dip_results = pd.read_csv(variables.DIP_MODELS_BLEND_PARAMS_PATH, index_col=0)
# calibrators = run_horizon_tasks(build_calibrators, "Calibrators per horizon")
# joblib.dump(calibrators, variables.ALL_MODELS_BEST_BLEND_PATH)


Export

In [14]:
def export_model_dict(spike_results, dip_results, calibrators):
    """Export ONLY optimization parameters (not trained models) to keep file size small."""
    
    # Load full_range_blend_alphas from CSV
    full_range_alphas_df = pd.read_csv(variables.FULL_RANGE_ALPHA_PATH)
    full_range_blend_alphas = full_range_alphas_df["Alpha"].values
    
    # Extract spike parameters
    spike_adjustment_modes = spike_results["spike_adjustment_mode"].values
    spike_source_modes = spike_results["spike_source_mode"].values
    spike_probability_thresholds = spike_results["probability_threshold"].values
    spike_probability_powers = spike_results["probability_power"].values
    spike_max_uplift_weights = spike_results["max_uplift_weight"].values
    
    # Extract dip parameters
    dip_adjustment_modes = dip_results["dip_adjustment_mode"].values
    dip_source_modes = dip_results["dip_source_mode"].values
    dip_probability_thresholds = dip_results["probability_threshold"].values
    dip_probability_powers = dip_results["probability_power"].values
    dip_max_reduction_weights = dip_results["max_reduction_weight"].values
    
    # Create NEW dictionary with ONLY parameters (no trained models)
    params_only = {
        "full_range_blend_alphas": full_range_blend_alphas,
        
        "spike_adjustment_modes": spike_adjustment_modes,
        "spike_source_modes": spike_source_modes,
        "spike_probability_thresholds": spike_probability_thresholds,
        "spike_probability_powers": spike_probability_powers,
        "spike_max_uplift_weights": spike_max_uplift_weights,
        
        "dip_adjustment_modes": dip_adjustment_modes,
        "dip_source_modes": dip_source_modes,
        "dip_probability_thresholds": dip_probability_thresholds,
        "dip_probability_powers": dip_probability_powers,
        "dip_max_reduction_weights": dip_max_reduction_weights,
        
        "calibrators": calibrators,
    }
    
    return params_only

spike_results = pd.read_csv(variables.SPIKE_MODELS_BLEND_PARAMS_PATH, index_col=0)
dip_results = pd.read_csv(variables.DIP_MODELS_BLEND_PARAMS_PATH, index_col=0)
calibrators = joblib.load(variables.ALL_MODELS_BEST_BLEND_PATH)
final_params = export_model_dict(spike_results, dip_results, calibrators)
joblib.dump(final_params, variables.FINAL_PARAMS_PATH)
print(f"Exported final parameters to {variables.FINAL_PARAMS_PATH}")

Exported final parameters to /home/daniel-davaris/Documents/NEM-Short-Term-Price-Forecasting/EPF/5_Model/Data/4_combine_models/final_params.joblib


In [15]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()


[release_memory] cleared 36 variable(s); kernel rss 0.28G, 8.6G RAM free now
